<a href="https://colab.research.google.com/github/chahelj/AILABREAL/blob/main/AILAB9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import copy
import re

class Term:
    def __init__(self, name, is_var=False):
        self.name = name
        self.is_var = is_var

    def __repr__(self):
        return f"?{self.name}" if self.is_var else self.name

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_var == other.is_var

    def __hash__(self):
        return hash((self.name, self.is_var))


class Literal:
    def __init__(self, predicate, args, negated=False):
        self.predicate = predicate
        self.args = args
        self.negated = negated

    def negate(self):
        return Literal(self.predicate, copy.deepcopy(self.args), not self.negated)

    def __repr__(self):
        prefix = "~" if self.negated else ""
        args_str = ", ".join(str(a) for a in self.args)
        return f"{prefix}{self.predicate}({args_str})"

    def __eq__(self, other):
        return (isinstance(other, Literal) and
                self.predicate == other.predicate and
                self.args == other.args and
                self.negated == other.negated)

    def __hash__(self):
        return hash((self.predicate, tuple(self.args), self.negated))


class Node:
    def __init__(self, type_, value=None, children=None):
        self.type = type_
        self.value = value
        self.children = children if children is not None else []

    def __repr__(self):
        if self.type == 'PREDICATE':
            args_str = ", ".join(f"?{arg}" if is_var else arg for arg, is_var in self.value[1])
            return f"{self.value[0]}({args_str})"
        elif self.type == 'NOT':
            return f"~({self.children[0]})"
        elif self.type in ('AND', 'OR', 'IMPLIES', 'IFF'):
            return f"({f' {self.type} '.join(str(c) for c in self.children)})"
        elif self.type in ('FORALL', 'EXISTS'):
            return f"({self.type} {self.value} . {self.children[0]})"
        return "UNKNOWN"


def tokenize(formula_str):
    tokens = []
    pattern = re.compile(r"(forall|exists|FORALL|EXISTS|~|&|\||=>|->|<=>|\(|\)|,|\w+)")
    for match in pattern.finditer(formula_str):
        token = match.group(1)
        if token.strip():
            tokens.append(token.strip())
    return tokens


class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None

    def consume(self):
        val = self.peek()
        self.pos += 1
        return val

    def parse(self):
        return self.parse_iff()

    def parse_iff(self):
        node = self.parse_implies()
        while self.peek() == '<=>':
            self.consume()
            right = self.parse_implies()
            node = Node('IFF', children=[node, right])
        return node

    def parse_implies(self):
        node = self.parse_or()
        while self.peek() in ('=>', '->'):
            self.consume()
            right = self.parse_or()
            node = Node('IMPLIES', children=[node, right])
        return node

    def parse_or(self):
        node = self.parse_and()
        while self.peek() == '|':
            self.consume()
            right = self.parse_and()
            node = Node('OR', children=[node, right])
        return node

    def parse_and(self):
        node = self.parse_unary()
        while self.peek() == '&':
            self.consume()
            right = self.parse_unary()
            node = Node('AND', children=[node, right])
        return node

    def parse_unary(self):
        token = self.peek()
        if token in ('forall', 'FORALL'):
            self.consume()
            var = self.consume()
            node = self.parse_unary()
            return Node('FORALL', value=var, children=[node])
        elif token in ('exists', 'EXISTS'):
            self.consume()
            var = self.consume()
            node = self.parse_unary()
            return Node('EXISTS', value=var, children=[node])
        elif token == '~':
            self.consume()
            node = self.parse_unary()
            return Node('NOT', children=[node])
        elif token == '(':
            self.consume()
            node = self.parse()
            self.consume()
            return node
        else:
            pred_name = self.consume()
            args = []
            if self.peek() == '(':
                self.consume()
                while True:
                    arg = self.consume()
                    is_var = arg[0].islower()
                    args.append((arg, is_var))
                    if self.peek() == ',':
                        self.consume()
                    else:
                        break
                self.consume()
            return Node('PREDICATE', value=(pred_name, args))


def parse_formula(formula_str):
    formula_str = formula_str.replace("->", "=>")
    tokens = tokenize(formula_str)
    parser = Parser(tokens)
    return parser.parse()


def eliminate_implications(node):
    if node.type == 'IMPLIES':
        left = eliminate_implications(node.children[0])
        right = eliminate_implications(node.children[1])
        return Node('OR', children=[Node('NOT', children=[left]), right])
    elif node.type == 'IFF':
        left = eliminate_implications(node.children[0])
        right = eliminate_implications(node.children[1])
        part1 = Node('OR', children=[Node('NOT', children=[left]), right])
        part2 = Node('OR', children=[Node('NOT', children=[right]), left])
        return Node('AND', children=[part1, part2])
    else:
        new_children = [eliminate_implications(c) for c in node.children]
        return Node(node.type, value=node.value, children=new_children)


def move_negation_inward(node):
    if node.type != 'NOT':
        return Node(node.type, value=node.value, children=[move_negation_inward(c) for c in node.children])
    child = node.children[0]
    if child.type == 'NOT':
        return move_negation_inward(child.children[0])
    elif child.type == 'AND':
        return Node('OR', children=[move_negation_inward(Node('NOT', children=[c])) for c in child.children])
    elif child.type == 'OR':
        return Node('AND', children=[move_negation_inward(Node('NOT', children=[c])) for c in child.children])
    elif child.type == 'FORALL':
        return Node('EXISTS', value=child.value, children=[move_negation_inward(Node('NOT', children=[child.children[0]]))])
    elif child.type == 'EXISTS':
        return Node('FORALL', value=child.value, children=[move_negation_inward(Node('NOT', children=[child.children[0]]))])
    elif child.type == 'PREDICATE':
        return node
    else:
        return node


def standardize_variables_node(node, mapping=None, counter=[0]):
    if mapping is None:
        mapping = {}
    if node.type in ('FORALL', 'EXISTS'):
        v = node.value
        counter[0] += 1
        new_v = f"{v}_{counter[0]}"
        new_mapping = mapping.copy()
        new_mapping[v] = new_v
        return Node(node.type, value=new_v, children=[standardize_variables_node(node.children[0], new_mapping, counter)])
    elif node.type == 'PREDICATE':
        pred_name, args = node.value
        new_args = []
        for arg, is_var in args:
            if is_var and arg in mapping:
                new_args.append((mapping[arg], True))
            else:
                new_args.append((arg, is_var))
        return Node('PREDICATE', value=(pred_name, new_args))
    else:
        return Node(node.type, value=node.value, children=[standardize_variables_node(c, mapping, counter) for c in node.children])


def skolemize(node, universal_vars=None, counter=[0]):
    if universal_vars is None:
        universal_vars = []
    if node.type == 'FORALL':
        return skolemize(node.children[0], universal_vars + [node.value], counter)
    elif node.type == 'EXISTS':
        counter[0] += 1
        sk_name = f"F_{counter[0]}"
        if not universal_vars:
            replacement = (sk_name, False)
        else:
            replacement = (f"{sk_name}({','.join(universal_vars)})", True)

        def substitute(n, target_var, repl):
            if n.type == 'PREDICATE':
                pred_name, args = n.value
                new_args = []
                for arg, is_var in args:
                    if is_var and arg == target_var:
                        if repl[1]:
                            terms_in_repl = [t.strip() for t in repl[0].split('(')[1].split(')')[0].split(',')]
                            for t in terms_in_repl:
                                new_args.append((t, True))
                        else:
                            new_args.append((repl[0], False))
                    else:
                        new_args.append((arg, is_var))
                return Node('PREDICATE', value=(pred_name, new_args))
            return Node(n.type, value=n.value, children=[substitute(c, target_var, repl) for c in n.children])

        substituted_child = substitute(node.children[0], node.value, replacement)
        return skolemize(substituted_child, universal_vars, counter)
    else:
        return Node(node.type, value=node.value, children=[skolemize(c, universal_vars, counter) for c in node.children])


def distribute_or(node):
    if node.type == 'AND':
        return Node('AND', children=[distribute_or(c) for c in node.children])
    elif node.type == 'OR':
        left = distribute_or(node.children[0])
        right = distribute_or(node.children[1])
        if left.type == 'AND':
            return Node('AND', children=[distribute_or(Node('OR', children=[c, right])) for c in left.children])
        elif right.type == 'AND':
            return Node('AND', children=[distribute_or(Node('OR', children=[left, c])) for c in right.children])
        else:
            return Node('OR', children=[left, right])
    else:
        return node


def collect_clauses(node):
    if node.type == 'AND':
        clauses = []
        for c in node.children:
            clauses.extend(collect_clauses(c))
        return clauses
    elif node.type == 'OR':
        left_clauses = collect_clauses(node.children[0])
        right_clauses = collect_clauses(node.children[1])
        res = []
        for lc in left_clauses:
            for rc in right_clauses:
                res.append(lc + rc)
        return res
    elif node.type == 'NOT':
        pred = node.children[0]
        pred_name, args = pred.value
        terms = [Term(arg, is_var) for arg, is_var in args]
        return [[Literal(pred_name, terms, negated=True)]]
    elif node.type == 'PREDICATE':
        pred_name, args = node.value
        terms = [Term(arg, is_var) for arg, is_var in args]
        return [[Literal(pred_name, terms, negated=False)]]
    return []


def convert_to_cnf(formula_str):
    tree = parse_formula(formula_str)
    tree = eliminate_implications(tree)
    tree = move_negation_inward(tree)
    tree = standardize_variables_node(tree)
    tree = skolemize(tree)
    tree = distribute_or(tree)
    return collect_clauses(tree)


def parse_literal(lit_str):
    lit_str = lit_str.strip()
    negated = False
    if lit_str.startswith("~") or lit_str.startswith("!"):
        negated = True
        lit_str = lit_str[1:].strip()

    match = re.match(r"^([A-Za-z0-9_]+)\((.*)\)$", lit_str)
    if not match:
        raise ValueError(f"Invalid literal syntax: '{lit_str}'. Expected format: Predicate(arg1, arg2)")

    predicate = match.group(1)
    raw_args = [arg.strip() for arg in match.group(2).split(",") if arg.strip()]

    terms = []
    for arg in raw_args:
        is_var = arg[0].islower()
        terms.append(Term(arg, is_var=is_var))

    return Literal(predicate, terms, negated=negated)


def parse_clause(clause_str):
    parts = re.split(r"\s*\|\s*|\s+OR\s+|\s+v\s+", clause_str.strip())
    return [parse_literal(part) for part in parts if part]


def standardize_variables(clause, counter):
    mapping = {}
    new_clause = []
    for lit in clause:
        new_args = []
        for arg in lit.args:
            if arg.is_var:
                if arg.name not in mapping:
                    mapping[arg.name] = Term(f"{arg.name}_{counter}", is_var=True)
                new_args.append(mapping[arg.name])
            else:
                new_args.append(arg)
        new_clause.append(Literal(lit.predicate, new_args, lit.negated))
    return new_clause


def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is None:
        return None
    elif x == y:
        return subst
    elif isinstance(x, Term) and x.is_var:
        return unify_var(x, y, subst)
    elif isinstance(y, Term) and y.is_var:
        return unify_var(y, x, subst)
    elif isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        if not x:
            return subst
        first_subst = unify(x[0], y[0], subst)
        if first_subst is None:
            return None
        return unify(x[1:], y[1:], first_subst)
    else:
        return None


def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    elif isinstance(x, Term) and x in subst:
        return unify(var, subst[x], subst)
    elif occur_check(var, x, subst):
        return None
    else:
        new_subst = subst.copy()
        new_subst[var] = x
        return new_subst


def occur_check(var, x, subst):
    if var == x:
        return True
    elif isinstance(x, Term) and x.is_var and x in subst:
        return occur_check(var, subst[x], subst)
    return False


def apply_subst(literal, subst):
    new_args = []
    for arg in literal.args:
        curr = arg
        while isinstance(curr, Term) and curr.is_var and curr in subst:
            curr = subst[curr]
        new_args.append(curr)
    return Literal(literal.predicate, new_args, literal.negated)


def resolve_clauses(clause1, clause2, step_counter):
    c1 = standardize_variables(clause1, f"a{step_counter}")
    c2 = standardize_variables(clause2, f"b{step_counter}")
    resolvents = []

    for lit1 in c1:
        for lit2 in c2:
            if lit1.predicate == lit2.predicate and lit1.negated != lit2.negated:
                subst = unify(lit1.args, lit2.args)
                if subst is not None:
                    new_clause = []
                    for l in c1:
                        if l != lit1:
                            new_clause.append(apply_subst(l, subst))
                    for l in c2:
                        if l != lit2:
                            applied = apply_subst(l, subst)
                            if applied not in new_clause:
                                new_clause.append(applied)
                    resolvents.append(frozenset(new_clause))
    return resolvents


def fol_resolution_prove(cnf_premises, negated_conclusion_clauses):
    clauses = set(frozenset(c) for c in cnf_premises)
    for nc in negated_conclusion_clauses:
        clauses.add(frozenset(nc))

    print("\nInitial Knowledge Base (CNF Clauses):")
    for c in clauses:
        print("  ", list(c))
    print("-" * 55)

    new_clauses = set()
    step_counter = 0

    while True:
        clause_list = list(clauses)
        n = len(clause_list)
        any_new_resolvents_found = False

        for i in range(n):
            for j in range(i + 1, n):
                step_counter += 1
                c1, c2 = clause_list[i], clause_list[j]

                resolvents = resolve_clauses(c1, c2, step_counter)

                for resolvent in resolvents:
                    if len(resolvent) == 0:
                        print(f"Resolved {list(c1)} and {list(c2)}  -->  [] (EMPTY CLAUSE)")
                        print("-" * 55)
                        print("SUCCESS: Contradiction found! Conclusion S is PROVED.")
                        return True

                    if resolvent not in clauses and resolvent not in new_clauses:
                        print(f"Resolved {list(c1)} and {list(c2)}  -->  {list(resolvent)}")
                        new_clauses.add(resolvent)
                        any_new_resolvents_found = True

        if not any_new_resolvents_found:
            print("-" * 55)
            print("FAILURE: No progress made. Conclusion S cannot be proved.")
            return False

        clauses.update(new_clauses)


if __name__ == "__main__":
    premises = [
        "forall x (Food(x) -> Likes(John, x))",
        "Food(Apple) & Food(Vegetables)",
        "forall x forall y ((Eats(y, x) & ~Killed(y)) -> Food(x))",
        "Eats(Anil, Peanuts) & Alive(Anil)",
        "forall x (Eats(Anil, x) -> Eats(Harry, x))",
        "forall x (Alive(x) -> ~Killed(x))",
        "forall x (~Killed(x) -> Alive(x))"
    ]
    conclusion = "Likes(John, Peanuts)"

    premises_cnf = []
    for p in premises:
        premises_cnf.extend(convert_to_cnf(p))

    negated_formula = f"~({conclusion})"
    negated_conclusion = convert_to_cnf(negated_formula)
    print("Chahel J, 1WN24CS066")
    print(f"-> Negated Conclusion (~S) added to KB: {negated_conclusion}")

    fol_resolution_prove(premises_cnf, negated_conclusion)


-> Negated Conclusion (~S) added to KB: [[~Likes(John, Peanuts)]]

Initial Knowledge Base (CNF Clauses):
   [Killed(?x_6), Alive(?x_6)]
   [~Likes(John, Peanuts)]
   [Food(Apple)]
   [Alive(Anil)]
   [Killed(?y_3), ~Eats(?y_3, ?x_2), Food(?x_2)]
   [Likes(John, ?x_1), ~Food(?x_1)]
   [Food(Vegetables)]
   [~Killed(?x_5), ~Alive(?x_5)]
   [Eats(Anil, Peanuts)]
   [Eats(Harry, ?x_4), ~Eats(Anil, ?x_4)]
-------------------------------------------------------
Resolved [Killed(?x_6), Alive(?x_6)] and [~Killed(?x_5), ~Alive(?x_5)]  -->  [Alive(?x_5_b7), ~Alive(?x_5_b7)]
Resolved [Killed(?x_6), Alive(?x_6)] and [~Killed(?x_5), ~Alive(?x_5)]  -->  [Killed(?x_5_b7), ~Killed(?x_5_b7)]
Resolved [~Likes(John, Peanuts)] and [Likes(John, ?x_1), ~Food(?x_1)]  -->  [~Food(Peanuts)]
Resolved [Food(Apple)] and [Likes(John, ?x_1), ~Food(?x_1)]  -->  [Likes(John, Apple)]
Resolved [Alive(Anil)] and [~Killed(?x_5), ~Alive(?x_5)]  -->  [~Killed(Anil)]
Resolved [Killed(?y_3), ~Eats(?y_3, ?x_2), Food(?x_2)] an

In [4]:
import copy
import re

class Term:
    def __init__(self, name, is_var=False):
        self.name = name
        self.is_var = is_var

    def __repr__(self):
        return f"?{self.name}" if self.is_var else self.name

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_var == other.is_var

    def __hash__(self):
        return hash((self.name, self.is_var))


class Literal:
    def __init__(self, predicate, args, negated=False):
        self.predicate = predicate
        self.args = args
        self.negated = negated

    def negate(self):
        return Literal(self.predicate, copy.deepcopy(self.args), not self.negated)

    def __repr__(self):
        prefix = "~" if self.negated else ""
        args_str = ", ".join(str(a) for a in self.args)
        return f"{prefix}{self.predicate}({args_str})"

    def __eq__(self, other):
        return (isinstance(other, Literal) and
                self.predicate == other.predicate and
                self.args == other.args and
                self.negated == other.negated)

    def __hash__(self):
        return hash((self.predicate, tuple(self.args), self.negated))


class Node:
    def __init__(self, type_, value=None, children=None):
        self.type = type_
        self.value = value
        self.children = children if children is not None else []

    def __repr__(self):
        if self.type == 'PREDICATE':
            args_str = ", ".join(f"?{arg}" if is_var else arg for arg, is_var in self.value[1])
            return f"{self.value[0]}({args_str})"
        elif self.type == 'NOT':
            return f"~({self.children[0]})"
        elif self.type in ('AND', 'OR', 'IMPLIES', 'IFF'):
            return f"({f' {self.type} '.join(str(c) for c in self.children)})"
        elif self.type in ('FORALL', 'EXISTS'):
            return f"({self.type} {self.value} . {self.children[0]})"
        return "UNKNOWN"


def tokenize(formula_str):
    tokens = []
    # Explicitly match operators, quantifiers, punctuation, or alphanumeric names
    pattern = re.compile(r"(forall|exists|FORALL|EXISTS|~|&|\||=>|->|<=>|\(|\)|,|\w+)")
    for match in pattern.finditer(formula_str):
        token = match.group(1)
        if token.strip():
            tokens.append(token.strip())
    return tokens


class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None

    def consume(self):
        val = self.peek()
        self.pos += 1
        return val

    def parse(self):
        return self.parse_iff()

    def parse_iff(self):
        node = self.parse_implies()
        while self.peek() == '<=>':
            self.consume()
            right = self.parse_implies()
            node = Node('IFF', children=[node, right])
        return node

    def parse_implies(self):
        node = self.parse_or()
        while self.peek() in ('=>', '->'):
            self.consume()
            right = self.parse_or()
            node = Node('IMPLIES', children=[node, right])
        return node

    def parse_or(self):
        node = self.parse_and()
        while self.peek() == '|':
            self.consume()
            right = self.parse_and()
            node = Node('OR', children=[node, right])
        return node

    def parse_and(self):
        node = self.parse_unary()
        while self.peek() == '&':
            self.consume()
            right = self.parse_unary()
            node = Node('AND', children=[node, right])
        return node

    def parse_unary(self):
        token = self.peek()
        if token in ('forall', 'FORALL'):
            self.consume()
            var = self.consume()
            node = self.parse_unary()
            return Node('FORALL', value=var, children=[node])
        elif token in ('exists', 'EXISTS'):
            self.consume()
            var = self.consume()
            node = self.parse_unary()
            return Node('EXISTS', value=var, children=[node])
        elif token == '~':
            self.consume()
            node = self.parse_unary()
            return Node('NOT', children=[node])
        elif token == '(':
            self.consume()
            node = self.parse()
            self.consume()
            return node
        else:
            pred_name = self.consume()
            args = []
            if self.peek() == '(':
                self.consume()
                while True:
                    arg = self.consume()
                    is_var = arg[0].islower()
                    args.append((arg, is_var))
                    if self.peek() == ',':
                        self.consume()
                    else:
                        break
                self.consume()
            return Node('PREDICATE', value=(pred_name, args))


def parse_formula(formula_str):
    # Standardize -> to => for parser
    formula_str = formula_str.replace("->", "=>")
    tokens = tokenize(formula_str)
    parser = Parser(tokens)
    return parser.parse()


def eliminate_implications(node):
    if node.type == 'IMPLIES':
        left = eliminate_implications(node.children[0])
        right = eliminate_implications(node.children[1])
        return Node('OR', children=[Node('NOT', children=[left]), right])
    elif node.type == 'IFF':
        left = eliminate_implications(node.children[0])
        right = eliminate_implications(node.children[1])
        part1 = Node('OR', children=[Node('NOT', children=[left]), right])
        part2 = Node('OR', children=[Node('NOT', children=[right]), left])
        return Node('AND', children=[part1, part2])
    else:
        new_children = [eliminate_implications(c) for c in node.children]
        return Node(node.type, value=node.value, children=new_children)


def move_negation_inward(node):
    if node.type != 'NOT':
        return Node(node.type, value=node.value, children=[move_negation_inward(c) for c in node.children])
    child = node.children[0]
    if child.type == 'NOT':
        return move_negation_inward(child.children[0])
    elif child.type == 'AND':
        return Node('OR', children=[move_negation_inward(Node('NOT', children=[c])) for c in child.children])
    elif child.type == 'OR':
        return Node('AND', children=[move_negation_inward(Node('NOT', children=[c])) for c in child.children])
    elif child.type == 'FORALL':
        return Node('EXISTS', value=child.value, children=[move_negation_inward(Node('NOT', children=[child.children[0]]))])
    elif child.type == 'EXISTS':
        return Node('FORALL', value=child.value, children=[move_negation_inward(Node('NOT', children=[child.children[0]]))])
    elif child.type == 'PREDICATE':
        return node
    else:
        return node


def standardize_variables_node(node, mapping=None, counter=[0]):
    if mapping is None:
        mapping = {}
    if node.type in ('FORALL', 'EXISTS'):
        v = node.value
        counter[0] += 1
        new_v = f"{v}_{counter[0]}"
        new_mapping = mapping.copy()
        new_mapping[v] = new_v
        return Node(node.type, value=new_v, children=[standardize_variables_node(node.children[0], new_mapping, counter)])
    elif node.type == 'PREDICATE':
        pred_name, args = node.value
        new_args = []
        for arg, is_var in args:
            if is_var and arg in mapping:
                new_args.append((mapping[arg], True))
            else:
                new_args.append((arg, is_var))
        return Node('PREDICATE', value=(pred_name, new_args))
    else:
        return Node(node.type, value=node.value, children=[standardize_variables_node(c, mapping, counter) for c in node.children])


def skolemize(node, universal_vars=None, counter=[0]):
    if universal_vars is None:
        universal_vars = []
    if node.type == 'FORALL':
        return skolemize(node.children[0], universal_vars + [node.value], counter)
    elif node.type == 'EXISTS':
        counter[0] += 1
        sk_name = f"F_{counter[0]}"
        if not universal_vars:
            replacement = (sk_name, False)
        else:
            replacement = (f"{sk_name}({','.join(universal_vars)})", True)

        def substitute(n, target_var, repl):
            if n.type == 'PREDICATE':
                pred_name, args = n.value
                new_args = []
                for arg, is_var in args:
                    if is_var and arg == target_var:
                        if repl[1]:
                            terms_in_repl = [t.strip() for t in repl[0].split('(')[1].split(')')[0].split(',')]
                            for t in terms_in_repl:
                                new_args.append((t, True))
                        else:
                            new_args.append((repl[0], False))
                    else:
                        new_args.append((arg, is_var))
                return Node('PREDICATE', value=(pred_name, new_args))
            return Node(n.type, value=n.value, children=[substitute(c, target_var, repl) for c in n.children])

        substituted_child = substitute(node.children[0], node.value, replacement)
        return skolemize(substituted_child, universal_vars, counter)
    else:
        return Node(node.type, value=node.value, children=[skolemize(c, universal_vars, counter) for c in node.children])


def distribute_or(node):
    if node.type == 'AND':
        return Node('AND', children=[distribute_or(c) for c in node.children])
    elif node.type == 'OR':
        left = distribute_or(node.children[0])
        right = distribute_or(node.children[1])
        if left.type == 'AND':
            return Node('AND', children=[distribute_or(Node('OR', children=[c, right])) for c in left.children])
        elif right.type == 'AND':
            return Node('AND', children=[distribute_or(Node('OR', children=[left, c])) for c in right.children])
        else:
            return Node('OR', children=[left, right])
    else:
        return node


def collect_clauses(node):
    if node.type == 'AND':
        clauses = []
        for c in node.children:
            clauses.extend(collect_clauses(c))
        return clauses
    elif node.type == 'OR':
        left_clauses = collect_clauses(node.children[0])
        right_clauses = collect_clauses(node.children[1])
        res = []
        for lc in left_clauses:
            for rc in right_clauses:
                res.append(lc + rc)
        return res
    elif node.type == 'NOT':
        pred = node.children[0]
        pred_name, args = pred.value
        terms = [Term(arg, is_var) for arg, is_var in args]
        return [[Literal(pred_name, terms, negated=True)]]
    elif node.type == 'PREDICATE':
        pred_name, args = node.value
        terms = [Term(arg, is_var) for arg, is_var in args]
        return [[Literal(pred_name, terms, negated=False)]]
    return []


def convert_to_cnf(formula_str):
    tree = parse_formula(formula_str)
    tree = eliminate_implications(tree)
    tree = move_negation_inward(tree)
    tree = standardize_variables_node(tree)
    tree = skolemize(tree)
    tree = distribute_or(tree)
    return collect_clauses(tree)


def parse_literal(lit_str):
    lit_str = lit_str.strip()
    negated = False
    if lit_str.startswith("~") or lit_str.startswith("!"):
        negated = True
        lit_str = lit_str[1:].strip()

    match = re.match(r"^([A-Za-z0-9_]+)\((.*)\)$", lit_str)
    if not match:
        raise ValueError(f"Invalid literal syntax: '{lit_str}'. Expected format: Predicate(arg1, arg2)")

    predicate = match.group(1)
    raw_args = [arg.strip() for arg in match.group(2).split(",") if arg.strip()]

    terms = []
    for arg in raw_args:
        is_var = arg[0].islower()
        terms.append(Term(arg, is_var=is_var))

    return Literal(predicate, terms, negated=negated)


def parse_clause(clause_str):
    parts = re.split(r"\s*\|\s*|\s+OR\s+|\s+v\s+", clause_str.strip())
    return [parse_literal(part) for part in parts if part]


def standardize_variables(clause, counter):
    mapping = {}
    new_clause = []
    for lit in clause:
        new_args = []
        for arg in lit.args:
            if arg.is_var:
                if arg.name not in mapping:
                    mapping[arg.name] = Term(f"{arg.name}_{counter}", is_var=True)
                new_args.append(mapping[arg.name])
            else:
                new_args.append(arg)
        new_clause.append(Literal(lit.predicate, new_args, lit.negated))
    return new_clause


def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is None:
        return None
    elif x == y:
        return subst
    elif isinstance(x, Term) and x.is_var:
        return unify_var(x, y, subst)
    elif isinstance(y, Term) and y.is_var:
        return unify_var(y, x, subst)
    elif isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        if not x:
            return subst
        first_subst = unify(x[0], y[0], subst)
        if first_subst is None:
            return None
        return unify(x[1:], y[1:], first_subst)
    else:
        return None


def unify_var(var, x, subst):
    if var in subst:
        return unify(subst[var], x, subst)
    elif isinstance(x, Term) and x in subst:
        return unify(var, subst[x], subst)
    elif occur_check(var, x, subst):
        return None
    else:
        new_subst = subst.copy()
        new_subst[var] = x
        return new_subst


def occur_check(var, x, subst):
    if var == x:
        return True
    elif isinstance(x, Term) and x.is_var and x in subst:
        return occur_check(var, subst[x], subst)
    return False


def apply_subst(literal, subst):
    new_args = []
    for arg in literal.args:
        curr = arg
        while isinstance(curr, Term) and curr.is_var and curr in subst:
            curr = subst[curr]
        new_args.append(curr)
    return Literal(literal.predicate, new_args, literal.negated)


def resolve_clauses(clause1, clause2, step_counter):
    c1 = standardize_variables(clause1, f"a{step_counter}")
    c2 = standardize_variables(clause2, f"b{step_counter}")
    resolvents = []

    for lit1 in c1:
        for lit2 in c2:
            if lit1.predicate == lit2.predicate and lit1.negated != lit2.negated:
                subst = unify(lit1.args, lit2.args)
                if subst is not None:
                    new_clause = []
                    for l in c1:
                        if l != lit1:
                            new_clause.append(apply_subst(l, subst))
                    for l in c2:
                        if l != lit2:
                            applied = apply_subst(l, subst)
                            if applied not in new_clause:
                                new_clause.append(applied)
                    resolvents.append(frozenset(new_clause))
    return resolvents


def fol_resolution_prove(cnf_premises, negated_conclusion_clauses):
    clauses = set(frozenset(c) for c in cnf_premises)
    for nc in negated_conclusion_clauses:
        clauses.add(frozenset(nc))

    print("\nInitial Knowledge Base (CNF Clauses):")
    for c in clauses:
        print("  ", list(c))
    print("-" * 55)

    new_clauses = set()
    step_counter = 0

    while True:
        clause_list = list(clauses)
        n = len(clause_list)
        any_new_resolvents_found = False

        for i in range(n):
            for j in range(i + 1, n):
                step_counter += 1
                c1, c2 = clause_list[i], clause_list[j]

                resolvents = resolve_clauses(c1, c2, step_counter)

                for resolvent in resolvents:
                    if len(resolvent) == 0:
                        print(f"Resolved {list(c1)} and {list(c2)}  -->  [] (EMPTY CLAUSE)")
                        print("-" * 55)
                        print("SUCCESS: Contradiction found! Conclusion S is PROVED.")
                        return True

                    if resolvent not in clauses and resolvent not in new_clauses:
                        print(f"Resolved {list(c1)} and {list(c2)}  -->  {list(resolvent)}")
                        new_clauses.add(resolvent)
                        any_new_resolvents_found = True

        if not any_new_resolvents_found:
            print("-" * 55)
            print("FAILURE: No progress made. Conclusion S cannot be proved.")
            return False

        clauses.update(new_clauses)


if __name__ == "__main__":
    premises = [
        "forall x (Food(x) -> Likes(John, x))",
        "Food(Apple) & Food(Vegetables)",
        "forall x forall y ((Eats(y, x) & ~Killed(y)) -> Food(x))",
        "Eats(Anil, Peanuts) & Alive(Anil)",
        "forall x (Eats(Anil, x) -> Eats(Harry, x))",
        "forall x (Alive(x) -> ~Killed(x))",
        "forall x (~Killed(x) -> Alive(x))"
    ]
    conclusion = "Likes(John, Peanuts)"

    premises_cnf = []
    for p in premises:
        premises_cnf.extend(convert_to_cnf(p))

    negated_formula = f"~({conclusion})"
    negated_conclusion = convert_to_cnf(negated_formula)
    print(f"-> Negated Conclusion (~S) added to KB: {negated_conclusion}")

    fol_resolution_prove(premises_cnf, negated_conclusion)


-> Negated Conclusion (~S) added to KB: [[~,()]]

Initial Knowledge Base (CNF Clauses):
   [,()]
   [~,()]
   [)(), ~,()]
   [)()]
   [,(), ~)()]
   [,(), ~,()]
   [~)()]
-------------------------------------------------------
Resolved [,()] and [~,()]  -->  [] (EMPTY CLAUSE)
-------------------------------------------------------
SUCCESS: Contradiction found! Conclusion S is PROVED.
